# generator-loss-fool-discriminator — worked example 2: Generator loss from raw logits

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `generator-loss-fool-discriminator`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The numerically stable form consumes D's pre-sigmoid logits via `F.binary_cross_entropy_with_logits`, which fuses sigmoid and BCE using `log1p(exp(·))`. With an all-ones target it computes the same non-saturating generator loss but stays finite for large-magnitude logits where the bare-probability form would overflow.

## Worked solution

We compute the generator loss directly from D's logits.

1. `d_logits` is D's output before the sigmoid, shape `(B,)`, unbounded.
2. Build the all-ones target as before: `targets = t.ones_like(d_logits)`. The generator still claims its fakes are real.
3. `F.binary_cross_entropy_with_logits(d_logits, targets)` applies sigmoid internally and computes BCE in one numerically stable pass. For a target of 1, the per-element loss is `softplus(-logit) = log(1 + exp(-logit))`.
4. We verify our value matches manually applying sigmoid then `binary_cross_entropy` for moderate logits, and print the loss for a strongly-fooled (large positive logit) batch to show it stays finite.

In [ ]:
import torch as t
import torch.nn.functional as F

t.manual_seed(1)

def generator_loss_logits(d_logits):
    targets = t.ones_like(d_logits)
    return F.binary_cross_entropy_with_logits(d_logits, targets)

logits = t.tensor([-1.0, 0.0, 2.0, 5.0])
manual = F.binary_cross_entropy(t.sigmoid(logits), t.ones_like(logits))
print('matches sigmoid+BCE:', bool(t.allclose(generator_loss_logits(logits), manual, atol=1e-5)))
print('large-logit loss finite:', float(generator_loss_logits(t.full((4,), 40.0))))